# Mental Health RAG Chatbot

1. Started with: Pre-trained LLM (meta-llama/Llama-3.3-70B-Instruct:fireworks-ai)
2. Applied: Fine-tuning on mental health datasets (none)
3. Built: RAG architecture with mental health knowledge base
4. Used: Prompt engineering to guide empathetic responses
5. Included: System prompts for safety and behavior guidelines

**Note:** this notebook builds and uses its own two FAISS stores (`general` PDF-derived knowledge + ICD-11 clinical data), kept in `./rag_system/`. It does **not** use script2's `mh-etl-data/faiss` index -- that one is a separate embedding space (forum-post embeddings for topic modeling), not clinical/guideline knowledge.


## Reference: pre-existing generic mental-health chatbots

LLM-based chatbots (newer generation -- 22 total identified): MindGuide, ChatCounselor, Replika, PsyChat, SMILE, Psy-LLM, Assistant-Instruct, CBT-LLM, MindWatch, EmLLM, Counsellor Chatbot.


## Data sources

1. Beyond Blue: https://www.beyondblue.org.au/mental-health/resource-library
2. Reachout: https://au.reachout.com/challenges-and-coping/abuse-and-violence/sexual-assault-support-services
3. First aid: https://www.mhfa.com.au/resources-support
4. CCI: https://www.cci.health.wa.gov.au/Resources/Looking-After-Others4
5. Sexual Assault Support Service (SASS): https://www.sass.org.au/resources
6. ICD-11 (professional diagnostic codes), DSM-5-TR (clinical case studies)

### Evaluation metrics

**Technical (computer-based):** Perplexity, ROUGE-L, BLEU, Distinct-1/2/3, BERTScore, Empathy%

**Human evaluation:** Helpfulness, Fluency, Relevance, Logic, Informativeness, Understanding, Consistency, Coherence, Empathy, Expertise, Engagement; counseling-specific: Direct Guidance, Approval and Reassurance, Restatement, Reflection, Listening, Interpretation, Self-disclosure (mirrors what real therapists do); reliability via Krippendorff's Alpha.


## 1. Scraping DSM-5-TR clinical cases

One-off acquisition step: downloads the DSM-5-TR clinical case PDF via an undetected Chrome session (the source requires a real browser session, not a plain HTTP request).

In [ ]:
import pandas as pd  # Kept as in original, though not used here
import time
import random
import json
import requests
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import undetected_chromedriver as uc

options = uc.ChromeOptions()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36')
options.set_capability("goog:loggingPrefs", {'performance': 'ALL'})

driver = uc.Chrome(options=options)

try:
    driver.get('https://psychiatryonline.org/doi/epdf/10.1176/appi.books.9781615375295')
    WebDriverWait(driver, 60).until(EC.presence_of_element_located((By.ID, 'viewer')))

    # Simulate human scrolling to trigger lazy-loaded content
    actions = ActionChains(driver)
    actions.move_by_offset(random.randint(100, 300), random.randint(100, 300)).perform()
    time.sleep(random.uniform(1, 3))
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
    time.sleep(random.uniform(2, 5))

    # If login is required, manually log in via the browser window, then press Enter here
    # input("Log in if necessary and press Enter to continue...")

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(random.uniform(3, 6))
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    time.sleep(5)

    logs = driver.get_log('performance')

    pdf_url = None
    for entry in logs:
        message = json.loads(entry['message'])['message']
        if message['method'] == 'Network.responseReceived':
            response = message['params'].get('response', {})
            url = response.get('url', '').lower()
            if '.pdf' in url or 'application/pdf' in response.get('mimeType', ''):
                pdf_url = response['url']
                print(f"Found potential PDF URL: {pdf_url}")
                break

    if not pdf_url:
        raise Exception("No PDF URL found in network logs. Try scrolling more or increasing wait time.")

    cookies = {c['name']: c['value'] for c in driver.get_cookies()}
    headers = {'User-Agent': options.arguments[-1].split('=')[1]}
    response = requests.get(pdf_url, cookies=cookies, headers=headers)

    if response.status_code == 200:
        with open('downloaded.pdf', 'wb') as f:
            f.write(response.content)
        print("PDF downloaded successfully as 'downloaded.pdf'")
    else:
        raise Exception(f"Failed to download PDF: Status code {response.status_code}")

finally:
    driver.quit()


## 2. Parse cases out of the extracted text

Splits the extracted PDF text into individual `CASE n.n` entries, then further splits each into description / discussion / diagnosis sections.

In [ ]:
import PyPDF2
import csv
import re


def clean_text(text):
    """Remove extra whitespace and empty lines, put all text on one line."""
    # Fix spaced-out words (e.g. "p s y c h i a t r y" -> "psychiatry")
    while re.search(r'\b(\w) (\w) (\w)', text):
        text = re.sub(r'\b(\w) (?=\w )', r'\1', text)

    text = re.sub(r' +', ' ', text)
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    return ' '.join(lines)


def parse_cases_from_file(txt_path):
    """Parse cases from extracted text file."""
    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Remove all "Suggested Reading" sections before processing
    text = re.sub(
        r'Suggested Reading.*?(?=CASE \d+\.\d+|\Z)',
        '', text, flags=re.DOTALL | re.IGNORECASE,
    )

    case_pattern = r'(CASE \d+\.\d+)'
    parts = re.split(case_pattern, text)

    cases = []
    for i in range(1, len(parts), 2):
        if i + 1 >= len(parts):
            break

        case_num = parts[i].strip()
        content = parts[i + 1].strip()
        lines = content.split('\n')

        # Extract case title (first non-empty line)
        case_title = ''
        author_line_idx = 0
        for idx, line in enumerate(lines):
            line = line.strip()
            if line:
                case_title = line
                author_line_idx = idx
                break

        # Skip author lines (usually 1-3 lines after title with names ending in degree)
        desc_start_idx = author_line_idx + 1
        while desc_start_idx < len(lines):
            line = lines[desc_start_idx].strip()
            if not line or re.search(r'\b(Ph\.?D\.?|M\.?D\.?|M\.?A\.?|D\.?O\.?)\b', line):
                desc_start_idx += 1
            else:
                break

        remaining_content = '\n'.join(lines[desc_start_idx:])

        description = ''
        discussion = ''
        diagnosis = ''

        if 'Discussion' in remaining_content:
            parts_disc = re.split(r'Discussion\s*\n', remaining_content, maxsplit=1)
            description = parts_disc[0].strip()
            after_discussion = parts_disc[1].strip() if len(parts_disc) > 1 else ''

            diag_match = re.search(r'(Diagnos[ei]s?)\s*\n', after_discussion)
            if diag_match:
                discussion = after_discussion[:diag_match.start()].strip()

                after_diagnosis = after_discussion[diag_match.end():].strip()
                # Handle line continuations with hyphens
                after_diagnosis = re.sub(r'-\s*\n\s*', '', after_diagnosis)
                # Handle regular line breaks within bullet points
                after_diagnosis = re.sub(r'(?<!•)\n(?!•)', ' ', after_diagnosis)

                diag_lines = [
                    line.strip() for line in after_diagnosis.split('\n')
                    if line.strip().startswith('•')
                ]
                diagnosis = ' '.join(diag_lines)
            else:
                discussion = after_discussion.strip()
                diagnosis = ''
        else:
            description = remaining_content.strip()
            discussion = ''
            diagnosis = ''

        cases.append({
            'case': case_title,
            'description': clean_text(description),
            'discussion': clean_text(discussion),
            'diagnosis': clean_text(diagnosis),
        })

    return cases


def write_to_csv(cases, csv_path='output_cases.csv'):
    """Write cases to CSV file."""
    with open(csv_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['case', 'description', 'discussion', 'diagnosis'])
        writer.writeheader()
        writer.writerows(cases)
    print(f'CSV file created successfully: {csv_path}')
    print(f'Total cases extracted: {len(cases)}')


if __name__ == "__main__":
    txt_path = 'extracted_text.txt'
    try:
        cases = parse_cases_from_file(txt_path)
        write_to_csv(cases)
    except FileNotFoundError:
        print(f"Error: Could not find {txt_path}")
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()


## 3. Anonymize patient identifiers

Strips patient names (`PERSON` entities) out of the extracted case text before this data goes anywhere near an embedding index, using a transformer NER model plugged into Presidio.

In [ ]:
from transformers import pipeline
from presidio_analyzer import EntityRecognizer, AnalyzerEngine, RecognizerResult
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import pandas as pd

# Full list of entities: https://microsoft.github.io/presidio/supported_entities/#list-of-supported-entities
DEFAULT_ANOYNM_ENTITIES = [
    "CREDIT_CARD", "CRYPTO", "DATE_TIME", "EMAIL_ADDRESS", "IBAN_CODE", "IP_ADDRESS",
    "NRP", "LOCATION", "PERSON", "PHONE_NUMBER", "MEDICAL_LICENSE", "URL", "ORGANIZATION", "NUMBER",
]


class TransformerRecognizer(EntityRecognizer):
    def __init__(self, model_id_or_path, mapping_labels, aggregation_strategy="simple",
                 supported_language="en", ignore_labels=["O"]):
        self.pipeline = pipeline(
            "token-classification", model=model_id_or_path,
            aggregation_strategy=aggregation_strategy, ignore_labels=ignore_labels,
        )
        self.label2presidio = mapping_labels
        super().__init__(supported_entities=list(self.label2presidio.values()), supported_language=supported_language)

    def load(self) -> None:
        """No loading is required."""
        pass

    def analyze(self, text: str, entities=None, nlp_artifacts=None):
        """Extracts entities using the Transformers pipeline."""
        results = []
        predicted_entities = self.pipeline(text)
        for e in predicted_entities:
            if e['entity_group'] not in self.label2presidio:
                continue
            converted_entity = self.label2presidio[e["entity_group"]]
            if converted_entity in entities or entities is None:
                results.append(RecognizerResult(
                    entity_type=converted_entity, start=e["start"], end=e["end"], score=e["score"],
                ))
        return results


mapping_labels = {"PER": "PERSON", 'LOC': 'LOCATION', 'ORG': "ORGANIZATION", "MISC": "NRP"}
configuration = {"nlp_engine_name": "spacy", "models": [{"lang_code": 'en', "model_name": "en_core_web_lg"}]}

to_keep = []  # words the model should NOT anonymize even if flagged
lang = 'en'

provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()

analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages="en")
transformers_recognizer = TransformerRecognizer("dslim/bert-base-NER", mapping_labels)
analyzer.registry.add_recognizer(transformers_recognizer)


def anonymize_text(text):
    if pd.isna(text):
        return ''
    analyzer_results = analyzer.analyze(text=text, entities=["PERSON"], allow_list=to_keep, language=lang)
    engine = AnonymizerEngine()
    operators = {"PERSON": OperatorConfig("replace", {"new_value": "[PATIENT]"})}
    result = engine.anonymize(text=text, analyzer_results=analyzer_results, operators=operators)
    return result.text


cases = pd.read_csv('output_cases.csv')
cases['description'] = cases['description'].apply(anonymize_text)
cases['discussion'] = cases['discussion'].apply(anonymize_text)
cases.to_csv('output_cases_anonymized.csv', index=False)


## 4. Environment check

In [ ]:
import platform
import psutil

print("Operating System:", platform.system())
print("OS Version:", platform.version())
print("OS Release:", platform.release())

print("\nCPU Info:")
print("Processor:", platform.processor())
print("Number of CPU Cores:", psutil.cpu_count(logical=True))

print("\nRAM Info:")
ram = psutil.virtual_memory()
print("Total RAM:", f"{ram.total / (1024 ** 3):.2f} GB")
print("Available RAM:", f"{ram.available / (1024 ** 3):.2f} GB")

print("\nDisk Info (C: drive):")
disk = psutil.disk_usage('C:\\')
print("Total Disk Space:", f"{disk.total / (1024 ** 3):.2f} GB")
print("Used Disk Space:", f"{disk.used / (1024 ** 3):.2f} GB")
print("Free Disk Space:", f"{disk.free / (1024 ** 3):.2f} GB")

try:
    import torch
    if torch.cuda.is_available():
        print("\nGPU Info:")
        print("GPU Name:", torch.cuda.get_device_name(0))
        print("GPU VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB")
    else:
        print("\nNo GPU detected or CUDA not available.")
except ImportError:
    print("\nTorch not installed; skipping GPU check.")

import faiss
print(faiss.get_num_gpus())


## 5. Model + API client setup

Uses the Hugging Face Inference Router (OpenAI-compatible API) rather than loading the LLM locally -- only the *embedder* runs locally/on-GPU.

In [ ]:
# # Install main packages (uncomment if needed, with -U for upgrades)
# %pip install -U transformers bitsandbytes accelerate peft datasets
# %pip install langchain-community faiss-gpu-cu12 sentence-transformers
# %pip install pypdf langchain-text-splitters langchain-chroma chromadb
# !pip install pyarrow==18.0.0 --force-reinstall  # if pyarrow/cudf conflicts arise


In [ ]:
import os
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
import faiss
from faiss import read_index, write_index
import torch
import logging
import nltk
from openai import OpenAI

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f"Using device: {device}")

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

# "meta-llama/Llama-3.1-405B-Instruct:fireworks-ai"
# "meta-llama/Llama-3.3-70B-Instruct:fireworks-ai"
model_name = "meta-llama/Llama-3.1-405B-Instruct:fireworks-ai"


## 6. General knowledge base: PDF parsing (LangChain) + embeddings + FAISS & Chroma

PDFs are now parsed and chunked with LangChain (`PyPDFLoader` + `RecursiveCharacterTextSplitter`) instead of `pdfplumber`, which also gives exact (not approximated) page numbers per chunk. The resulting chunks are embedded once with the same SapBERT encoder and written to **both** a FAISS index and a Chroma collection, so the two backends are directly comparable. The chatbot's retrieval (section 9) defaults to FAISS; Chroma is available via `vector_backend='chroma'` if you want to compare it.

In [ ]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import logging
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# LangChain: PDF parsing + chunking + Chroma vector store
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.embeddings import Embeddings as LCEmbeddings

nltk.download('vader_lexicon', quiet=True)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f"Using device: {device}")

embedder = SentenceTransformer('cambridgeltl/SapBERT-from-PubMedBERT-fulltext', device=device)


class SentenceTransformerEmbeddings(LCEmbeddings):
    """LangChain-compatible wrapper around the SentenceTransformer `embedder`
    above, so Chroma (and anything else in LangChain) embeds text with the
    exact same model + normalization used for the FAISS index -- keeping the
    two vector stores comparable rather than using different embedding spaces."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return embedder.encode(
            texts, batch_size=128, device=device, normalize_embeddings=True,
        ).tolist()

    def embed_query(self, text: str) -> list[float]:
        return embedder.encode(
            [text], device=device, normalize_embeddings=True,
        )[0].tolist()


st_embeddings = SentenceTransformerEmbeddings()

# Configurable paths (change category as needed)
category = 'general'  # 'anxiety', 'depression', 'ptsd', 'suicide', general
pdf_dir = f'./rag_system/{category}/'
rag_chunks_csv_path = f'./rag_system/{category}/chunks.csv'
rag_embeddings_path = f'./rag_system/{category}/chunk_embeddings.npy'
rag_index_path = f'./rag_system/{category}/faiss_index.index'
rag_chroma_dir = f'./rag_system/{category}/chroma_store'

os.makedirs(os.path.dirname(rag_chunks_csv_path), exist_ok=True)

# chunk_size/overlap are in characters (LangChain's splitter unit), roughly
# equivalent to the previous 200-word / 40-word overlap chunking, but this
# also keeps sentence/paragraph boundaries where possible instead of cutting
# mid-sentence at a fixed word count.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def extract_and_chunk_pdf(pdf_path, splitter=text_splitter):
    """Load a PDF with LangChain's PyPDFLoader (one Document per page) and
    split into overlapping chunks with the shared text_splitter, preserving
    the true page number as metadata (PyPDFLoader gives exact pages, unlike
    the previous word-count approximation)."""
    chunks = []
    try:
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()  # one Document per page; metadata['page'] is 0-indexed
        split_docs = splitter.split_documents(pages)

        for doc in split_docs:
            chunks.append({
                'content': doc.page_content,
                'source_pdf': os.path.basename(pdf_path),
                'page_start': doc.metadata.get('page', 0) + 1,  # 1-indexed, exact page
                'category': category,
            })

        logger.info(f"Extracted {len(chunks)} chunks from {pdf_path}")
    except Exception as e:
        logger.error(f"Error processing {pdf_path}: {e}")
    return chunks


def compute_sentiment_labels(texts):
    """Compound VADER score bucketed into positive/neutral/negative."""
    sia = SentimentIntensityAnalyzer()

    def get_sentiment(text):
        score = sia.polarity_scores(text)['compound']
        if score > 0.05:
            return 'positive'
        elif score < -0.05:
            return 'negative'
        return 'neutral'

    return texts.apply(get_sentiment)


# Step 1: Process all PDFs (via LangChain's PyPDFLoader) and save chunks to CSV
if os.path.exists(rag_chunks_csv_path):
    logger.info(f"Loading existing chunks from {rag_chunks_csv_path}")
    chunks_df = pd.read_csv(rag_chunks_csv_path)
else:
    all_chunks = []
    for filename in os.listdir(pdf_dir):
        if filename.endswith('.pdf'):
            pdf_path = os.path.join(pdf_dir, filename)
            all_chunks.extend(extract_and_chunk_pdf(pdf_path))

    if not all_chunks:
        raise ValueError(f"No chunks extracted from PDFs in {pdf_dir}")

    chunks_df = pd.DataFrame(all_chunks)
    chunks_df['chunk_id'] = range(len(chunks_df))
    chunks_df.to_csv(rag_chunks_csv_path, index=False)
    logger.info(f"Saved {len(chunks_df)} chunks to {rag_chunks_csv_path}")

if 'sentiment' not in chunks_df.columns:
    logger.info("Computing sentiment for chunks...")
    chunks_df['sentiment'] = compute_sentiment_labels(chunks_df['content'])
    chunks_df.to_csv(rag_chunks_csv_path, index=False)
    logger.info("Sentiment added and CSV updated.")

# Step 2: Load or compute embeddings (same SapBERT model feeds both FAISS and Chroma)
if os.path.exists(rag_embeddings_path):
    logger.info("Loading precomputed embeddings...")
    chunk_embeddings_np = np.load(rag_embeddings_path)
else:
    logger.info("Computing embeddings...")
    chunk_embeddings = embedder.encode(
        chunks_df['content'].tolist(), batch_size=128, show_progress_bar=True,
        convert_to_tensor=True, device=device, normalize_embeddings=True,
    )
    chunk_embeddings_np = chunk_embeddings.cpu().numpy()
    np.save(rag_embeddings_path, chunk_embeddings_np)
    logger.info(f"Embeddings saved to {rag_embeddings_path}")

# Step 3: Build/Load FAISS Index -- this is the store the chatbot actually
# queries at runtime (see the retrieval engine section, `vector_backend='faiss'`).
d = chunk_embeddings_np.shape[1]
if os.path.exists(rag_index_path):
    logger.info("Loading existing FAISS index...")
    faiss_index = faiss.read_index(rag_index_path)
else:
    logger.info("Building FAISS index...")
    index = faiss.IndexFlatIP(d)  # Inner product for normalized vectors
    index.add(chunk_embeddings_np)
    faiss.write_index(index, rag_index_path)
    logger.info(f"FAISS index saved to {rag_index_path}")

# Step 4: Build/Load a Chroma vector store with the SAME chunks + embedder.
# This exists so FAISS vs. Chroma can be compared later -- it is built here
# but the chatbot's retrieval below defaults to FAISS and only switches to
# this Chroma store if `vector_backend='chroma'` is passed explicitly.
if os.path.exists(rag_chroma_dir) and os.listdir(rag_chroma_dir):
    logger.info("Loading existing Chroma store...")
    chroma_store = Chroma(
        persist_directory=rag_chroma_dir, embedding_function=st_embeddings,
        collection_name=f"{category}_chunks",
    )
else:
    logger.info("Building Chroma store...")
    chroma_store = Chroma.from_texts(
        texts=chunks_df['content'].tolist(),
        embedding=st_embeddings,
        metadatas=chunks_df[['source_pdf', 'page_start', 'category', 'sentiment']].to_dict('records'),
        ids=[str(i) for i in chunks_df['chunk_id']],
        collection_name=f"{category}_chunks",
        persist_directory=rag_chroma_dir,
    )
    logger.info(f"Chroma store persisted to {rag_chroma_dir}")


## 7. ICD-11 knowledge base

Fetches Chapter 6 (mental, behavioural, neurodevelopmental disorders) from the WHO ICD-11 API, keeping only leaf-node disorders (skips chapter/block headers), then builds its own separate FAISS store.

In [ ]:
import requests
import json
from dotenv import load_dotenv
load_dotenv()

icd_dir = './rag_system/ICD-11_Data/'
icd_json_path = f'{icd_dir}/icd_disorders.json'
icd_chunks_csv_path = f'{icd_dir}/icd_chunks.csv'
icd_embeddings_path = f'{icd_dir}/icd_embeddings.npy'
icd_index_path = f'{icd_dir}/faiss_index.index'

os.makedirs(icd_dir, exist_ok=True)

CLIENT_ID = os.environ.get('ICD_CLIENT_ID')
CLIENT_SECRET = os.environ.get('ICD_CLIENT_SECRET')
SCOPE = 'icdapi_access'
GRANT_TYPE = 'client_credentials'
TOKEN_ENDPOINT = 'https://icdaccessmanagement.who.int/connect/token'
API_BASE = 'https://id.who.int/icd'
RELEASE = '2025-01'  # Update to the latest release if needed
LINEARIZATION = 'mms'  # Mortality and Morbidity Statistics
CHAPTER_ENTITY_ID = '334423054'  # Chapter 6: Mental, behavioural or neurodevelopmental disorders


def get_access_token():
    if not CLIENT_ID or not CLIENT_SECRET:
        raise ValueError("ICD_CLIENT_ID and ICD_CLIENT_SECRET must be set in environment variables.")
    payload = {
        'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET,
        'scope': SCOPE, 'grant_type': GRANT_TYPE,
    }
    response = requests.post(TOKEN_ENDPOINT, data=payload)
    response.raise_for_status()
    return response.json()['access_token']


def fetch_entity(entity_id, token, is_foundation=False):
    headers = {
        'Authorization': f'Bearer {token}', 'Accept': 'application/json',
        'Accept-Language': 'en', 'API-Version': 'v2',
    }
    url = f'{API_BASE}/entity/{entity_id}' if is_foundation else f'{API_BASE}/release/11/{RELEASE}/{LINEARIZATION}/{entity_id}'
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()


def collect_disorders(parent_id, token, collected=None, depth=0, max_depth=10):
    """Recursively collect all leaf-node disorders under a parent entity."""
    if collected is None:
        collected = []
    if depth > max_depth:
        logger.warning(f"Max depth reached for {parent_id}")
        return collected

    try:
        data = fetch_entity(parent_id, token)

        code_ = data.get('code', '')
        fully_specified_name = data.get('title', {}).get('@value', '') if isinstance(data.get('title'), dict) else data.get('title', '')
        description = data.get('definition', {}).get('@value', '') if isinstance(data.get('definition'), dict) else data.get('definition', '')

        exclusions = [
            excl.get('label', {}).get('@value', '') if isinstance(excl.get('label'), dict) else excl.get('label', '')
            for excl in data.get('exclusion', [])
        ]
        index_terms = [
            term.get('label', {}).get('@value', '') if isinstance(term.get('label'), dict) else term.get('label', '')
            for term in data.get('indexTerm', [])
        ]

        has_children = bool(data.get('child'))

        if not has_children and code_ and fully_specified_name and not fully_specified_name.startswith(('Chapter', 'Block')):
            collected.append({
                'code': code_,
                'fully_specified_name': fully_specified_name,
                'description': description,
                'exclusions': '; '.join(exclusions),
                'all_index_terms': '; '.join(index_terms),
                'entity_id': parent_id,
            })
            logger.info(f"Collected (leaf): {code_} - {fully_specified_name}")

        if has_children:
            for child_uri in data['child']:
                child_id = child_uri.split('/')[-1]
                collect_disorders(child_id, token, collected, depth + 1, max_depth)

    except requests.HTTPError as e:
        logger.error(f"Error fetching {parent_id}: {e}")

    return collected


# Step 1: Fetch and save ICD-11 data to CSV and JSON
if os.path.exists(icd_chunks_csv_path):
    logger.info(f"Loading existing ICD data from {icd_chunks_csv_path}")
    icd_df = pd.read_csv(icd_chunks_csv_path)
elif os.path.exists(icd_json_path):
    logger.info(f"Loading existing ICD data from {icd_json_path}")
    with open(icd_json_path, 'r') as f:
        disorders = json.load(f)
    icd_df = pd.DataFrame(disorders)
    icd_df.to_csv(icd_chunks_csv_path, index=False)
    logger.info(f"Saved CSV from JSON to {icd_chunks_csv_path}")
else:
    token = get_access_token()
    disorders = collect_disorders(CHAPTER_ENTITY_ID, token)
    if not disorders:
        raise ValueError("No disorders collected from ICD-11 API.")

    with open(icd_json_path, 'w') as f:
        json.dump(disorders, f, indent=4)
    logger.info(f"Saved disorders to {icd_json_path}")

    icd_df = pd.DataFrame(disorders)
    icd_df.to_csv(icd_chunks_csv_path, index=False)
    logger.info(f"Saved {len(icd_df)} disorders to {icd_chunks_csv_path}")

if 'content' not in icd_df.columns:
    icd_df['content'] = icd_df.apply(
        lambda row: f" {row['fully_specified_name']} {row['description']} {row['all_index_terms']}", axis=1,
    )

if 'sentiment' not in icd_df.columns:
    logger.info("Computing sentiment for ICD entries...")
    icd_df['sentiment'] = compute_sentiment_labels(icd_df['content'])
    icd_df.to_csv(icd_chunks_csv_path, index=False)
    logger.info("Sentiment added and CSV updated.")

# Step 2: Load or compute embeddings
if os.path.exists(icd_embeddings_path):
    logger.info("Loading precomputed embeddings...")
    icd_embeddings_np = np.load(icd_embeddings_path)
else:
    logger.info("Computing embeddings...")
    icd_embeddings = embedder.encode(
        icd_df['content'].tolist(), batch_size=128, show_progress_bar=True,
        convert_to_tensor=True, device=device, normalize_embeddings=True,
    )
    icd_embeddings_np = icd_embeddings.cpu().numpy()
    np.save(icd_embeddings_path, icd_embeddings_np)
    logger.info(f"Embeddings saved to {icd_embeddings_path}")

# Step 3: Build/Load FAISS Index
if os.path.exists(icd_index_path):
    logger.info("Loading existing FAISS index...")
    icd_faiss_index = faiss.read_index(icd_index_path)
else:
    logger.info("Building FAISS index...")
    d = icd_embeddings_np.shape[1]
    icd_faiss_index = faiss.IndexFlatIP(d)
    icd_faiss_index.add(icd_embeddings_np)
    write_index(icd_faiss_index, icd_index_path)
    logger.info(f"FAISS index saved to {icd_index_path}")


## 8. Load both RAG resources into memory

In [ ]:
# Categories (assuming lowercase for paths, but adjust if needed)
# 'anxiety', 'depression', 'ptsd', 'suicide', 'general'
categories = ['general']  # Include 'general' as fallback

category_resources = {}
for cat in categories:
    chunks_csv = f'./rag_system/{cat}/chunks.csv'
    emb_path = f'./rag_system/{cat}/chunk_embeddings.npy'
    idx_path = f'./rag_system/{cat}/faiss_index.index'
    chroma_dir = f'./rag_system/{cat}/chroma_store'

    if os.path.exists(chunks_csv) and os.path.exists(emb_path) and os.path.exists(idx_path):
        chunks_df = pd.read_csv(chunks_csv)
        if 'sentiment' not in chunks_df.columns:
            logger.info(f"Computing sentiment for {cat} chunks...")
            chunks_df['sentiment'] = compute_sentiment_labels(chunks_df['content'])
            chunks_df.to_csv(chunks_csv, index=False)
            logger.info(f"Sentiment added for {cat} and CSV updated.")

        # Chroma is optional -- only loaded so `vector_backend='chroma'` works
        # if someone wants to compare it; the chatbot defaults to FAISS below.
        cat_chroma = None
        if os.path.exists(chroma_dir) and os.listdir(chroma_dir):
            try:
                cat_chroma = Chroma(
                    persist_directory=chroma_dir, embedding_function=st_embeddings,
                    collection_name=f"{cat}_chunks",
                )
                logger.info(f"Loaded Chroma store for category: {cat}")
            except Exception as e:
                logger.warning(f"Could not load Chroma store for {cat}: {e}")

        category_resources[cat] = {
            'df': chunks_df,
            'embeddings': np.load(emb_path),
            'index': read_index(idx_path),
            'chroma': cat_chroma,
        }
        logger.info(f"Loaded resources for category: {cat}")
    else:
        logger.warning(f"Missing resources for category: {cat}. Skipping.")

# ICD-11 resources
icd_dir = './rag_system/ICD-11_Data/'
icd_chunks_csv_path = f'{icd_dir}/icd_chunks.csv'
icd_embeddings_path = f'{icd_dir}/icd_embeddings.npy'
icd_index_path = f'{icd_dir}/faiss_index.index'

if os.path.exists(icd_chunks_csv_path):
    icd_df = pd.read_csv(icd_chunks_csv_path)
    if 'content' not in icd_df.columns:
        icd_df['content'] = icd_df.apply(
            lambda row: f"{row['code']} {row['fully_specified_name']} {row['description']} {row['exclusions']} {row['all_index_terms']}", axis=1,
        )
        icd_df.to_csv(icd_chunks_csv_path, index=False)
else:
    raise FileNotFoundError(f"ICD-11 CSV not found at {icd_chunks_csv_path}. Please run the ICD-11 prep cell first.")

if os.path.exists(icd_embeddings_path):
    icd_embeddings_np = np.load(icd_embeddings_path)
else:
    raise FileNotFoundError(f"ICD-11 embeddings not found at {icd_embeddings_path}.")

if os.path.exists(icd_index_path):
    icd_index = read_index(icd_index_path)
else:
    raise FileNotFoundError(f"ICD-11 FAISS index not found at {icd_index_path}.")


## 9. Retrieval engine

Query rewriting (for multi-turn retrieval), category retrieval with 3 selectable query strategies (`original` / `multi` / `hyde`), diversity-aware reranking, and the LLM call helpers. Retrieval also takes a `vector_backend` argument (`'faiss'` by default, `'chroma'` as a switchable alternative) so the FAISS and Chroma stores built in section 6 can be queried the same way.

In [ ]:
# API-based generation for RAG (single prompt, no chat history)
def generate_with_llm_rag(prompt: str, max_tokens: int = 120):
    messages = [{"role": "user", "content": prompt}]
    completion = client.chat.completions.create(
        model=model_name, messages=messages, max_tokens=max_tokens,
        temperature=0.8, top_p=0.95, frequency_penalty=1.2,
    )
    return completion.choices[0].message.content.strip()


# API-based generation for the chatbot (full message list)
def generate_with_llm(messages: list[dict], max_tokens: int = 200) -> str:
    completion = client.chat.completions.create(
        model=model_name, messages=messages, max_tokens=max_tokens,
        temperature=0.8, top_p=0.98, frequency_penalty=1.0,
    )
    return completion.choices[0].message.content.strip()


def rewrite_query_with_history(query: str, history: list[dict], max_turns: int = 6) -> str:
    """Rewrite the latest user message into a standalone, context-complete
    query using recent conversation history -- for RETRIEVAL only (FAISS
    search / HyDE generation). NOT used for display, and NOT used for the
    final response-generation prompt (which still uses the raw `query`, so
    the assistant replies to what the user actually typed, not a rewritten
    version of it).

    Without this, follow-up messages like "what about medication?" get
    embedded and searched with zero context about what "it" refers to,
    which silently degrades retrieval quality on multi-turn conversations.
    `history` should be the list of {'role', 'content'} dicts BEFORE the
    current query is appended.
    """
    if not history:
        return query

    recent = history[-(max_turns * 2):]  # max_turns pairs = max_turns*2 messages
    history_text = "\n".join(f"{turn['role']}: {turn['content']}" for turn in recent)

    prompt = f"""Given the conversation history and the user's latest message, rewrite the latest message as a single standalone query that includes any necessary context from the history (e.g. resolve "it"/"that"/vague follow-ups into what they refer to). Do not answer the message. Preserve the user's original tone and intent. If the latest message is already standalone, return it unchanged. Output ONLY the rewritten query, nothing else.

Conversation history:
{history_text}

Latest message: {query}

Standalone query:"""

    try:
        rewritten = generate_with_llm_rag(prompt, max_tokens=80).strip()
        # Guard against empty/degenerate rewrites (e.g. the LLM answering
        # the question instead of rewriting it, or returning nothing)
        if not rewritten or len(rewritten.split()) < 2:
            return query
        return rewritten
    except Exception as e:
        logger.warning(f"Query rewrite failed, falling back to raw query: {e}")
        return query


def category_retrieve(category: str, query: str, method: str = 'original', k: int = 10, vector_backend: str = 'faiss') -> list[dict]:
    """Retrieve top-k chunks for a category, using one of three query
    strategies: 'original' (embed the query as-is), 'multi' (generate query
    variants and pool results), or 'hyde' (embed a hypothetical answer
    document instead of the query itself -- often retrieves better for
    short/vague queries).

    `vector_backend` picks which vector store is actually searched:
    'faiss' (default -- what the chatbot uses in production) or 'chroma'
    (built alongside FAISS in the ingestion cell, useful for comparing the
    two backends on the same chunks/embeddings). Falls back to FAISS with a
    warning if a Chroma store isn't loaded for the category.
    """
    if category not in category_resources:
        logger.warning(f"Category '{category}' not found. Falling back to 'general'.")
        category = 'general'
        if category not in category_resources:
            raise ValueError("No 'general' resources available as fallback.")

    res = category_resources[category]
    chunks_df = res['df']
    cat_index = res['index']
    cat_chroma = res.get('chroma')

    if vector_backend == 'chroma' and cat_chroma is None:
        logger.warning(f"No Chroma store loaded for '{category}'; falling back to FAISS.")
        vector_backend = 'faiss'

    queries = [query]

    if method == 'multi':
        prompt = f"""
            Strictly follow: Output EXACTLY 5 unique variant queries similar to '{query}', one per line.
            Preserve key elements (events, causes, feelings).
            No reasoning, no steps, no examples, no introductions, no extra text at all.
            Directly start with the first query.

            Example output format (do not include this in output):
            Variant1
            Variant2
            Variant3
            Variant4
            Variant5
            """
        variants_response = generate_with_llm_rag(prompt)
        variants = [
            line.strip() for line in variants_response.split('\n')
            if line.strip() and len(line.split()) > 2
            and not re.match(r'^(#|Step|Example|For reference|The|To|Output|Preserve|No|Directly)', line, re.I)
        ]
        queries = list(set(v for v in variants if v.lower() != query.lower()))[:5]
        if not queries or len(queries) < 3:
            queries = [query, f"{query} coping strategies", f"{query} emotional support"]
        logger.info(f"Generated multi-queries: {queries}")

    elif method == 'hyde':
        prompt = f"""
            Generate a concise hypothetical answer document for the query '{query}'.
            Output only the document text, without any introductions, explanations, numbering,
            or extra formatting.
            """
        hyde_response = generate_with_llm_rag(prompt)
        hyde_doc = ' '.join(line.strip() for line in hyde_response.split('\n') if line.strip()).replace('Document:', '').strip()
        queries = [hyde_doc]
        logger.info(f"Generated HyDE document: {hyde_doc[:200]}...")

    all_results = {}

    if vector_backend == 'chroma':
        for q in queries:
            hits = cat_chroma.similarity_search_with_relevance_scores(q, k=k)
            for doc, score in hits:
                # Chroma doesn't hand back the same stable integer id FAISS does,
                # so dedupe on content instead.
                key = doc.page_content
                if key not in all_results or score > all_results[key]['score']:
                    all_results[key] = {
                        'content': doc.page_content,
                        'source_pdf': doc.metadata.get('source_pdf'),
                        'page_start': doc.metadata.get('page_start'),
                        'sentiment': doc.metadata.get('sentiment'),
                        'score': score,
                    }
    else:  # 'faiss' (default)
        query_embs = embedder.encode(queries, convert_to_tensor=True, device=device, normalize_embeddings=True, show_progress_bar=False).cpu().numpy()
        for q_emb in query_embs:
            q_emb = q_emb.reshape(1, -1)
            distances, indices = cat_index.search(q_emb, k=k)
            for i in range(len(distances[0])):
                idx = indices[0][i]
                if idx == -1:
                    continue
                score = distances[0][i]
                if idx not in all_results or score > all_results[idx]['score']:
                    row = chunks_df.iloc[idx]
                    all_results[idx] = {
                        'content': row['content'], 'source_pdf': row['source_pdf'],
                        'page_start': row['page_start'], 'sentiment': row['sentiment'], 'score': score,
                    }

    return sorted(all_results.values(), key=lambda x: x['score'], reverse=True)[:k]


def rerank_results(results: list[dict]) -> list[dict]:
    """Sort by sentiment (positive-first) then score, with a diversity
    filter that drops results too similar (cosine > 0.85) to one already kept."""
    sentiment_order = {'positive': 0, 'neutral': 1, 'negative': 2}
    sorted_results = sorted(results, key=lambda x: (sentiment_order.get(x['sentiment'], 3), -x['score']))

    diverse = [sorted_results[0]]
    for res in sorted_results[1:]:
        res_emb = embedder.encode(res['content'], convert_to_tensor=True, device=device, show_progress_bar=False).cpu().numpy()
        similar = False
        for d in diverse:
            d_emb = embedder.encode(d['content'], convert_to_tensor=True, device=device, show_progress_bar=False).cpu().numpy()
            sim = np.dot(res_emb.flatten(), d_emb.flatten()) / (np.linalg.norm(res_emb) * np.linalg.norm(d_emb))
            if sim > 0.85:
                similar = True
                break
        if not similar:
            diverse.append(res)
    return diverse[:5]


def full_rag_workflow(query: str, method: str = 'original', vector_backend: str = 'faiss') -> list[dict]:
    """Retrieve + rerank for a query. Category is pinned to 'general' --
    change this line if dynamic category routing is added later.
    `vector_backend` defaults to 'faiss' (what the chatbot uses); pass
    'chroma' to retrieve from the Chroma store instead, e.g. for a
    side-by-side comparison of the two backends."""
    category = 'general'  # Comment out if you want dynamic category
    retrieved = category_retrieve(category, query, method=method, vector_backend=vector_backend)
    return rerank_results(retrieved)


# example_query = "my dog pass away"
# results = full_rag_workflow(example_query, method='hyde')  # FAISS (default)
# results_chroma = full_rag_workflow(example_query, method='hyde', vector_backend='chroma')  # Chroma, for comparison
# for i, res in enumerate(results):
#     print(f"Top {i+1}: Score: {res['score']:.4f}, Sentiment: {res['sentiment']}, Source: {res['source_pdf']}")
#     print(f"Content snippet: {res['content'][:300]}...\n")


## 10. Shared chatbot-reply logic

This is the piece that used to be copy-pasted -- with small drifting differences -- across the CLI runner, the GUI class, and two evaluation harnesses: the system prompt, `trim_to_complete_sentence`, and the full "rewrite query -> check ICD-11 -> retrieve -> build prompt -> call LLM -> trim" sequence.

It's consolidated here into one `SYSTEM_PROMPT` constant and one `generate_chatbot_reply()` function that every caller below now shares. `generate_chatbot_reply()` does **not** mutate or append to `history` -- callers append the new turn themselves afterward, since the CLI/GUI keep a running history across turns while the evaluation harnesses reset history per query group.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SYSTEM_PROMPT = """
    You are a compassionate and empathetic mental health assistant, adapting your role dynamically—like a caring friend for casual chats or a thoughtful guide for deeper reflections—to keep interactions fresh. Your responses should:
    - Analyze input empathetically, tailoring to query, history, and guidelines without rigid patterns (e.g., vary openings with questions, shared observations, or gentle reflections first).
    - Respond conversationally: Mix tones (warm, curious, uplifting) and structures (e.g., short paragraphs, bullets for suggestions, questions, or analogies) to avoid repetition and enhance diversity.
    - Weave in guidelines naturally (e.g., coping ideas as "One approach I've heard helps is...").
    - If relevant matching disorders from ICD-11 are provided in the user prompt, integrate them early as non-diagnostic possibilities to explore (e.g., "Some experiences like yours remind me of [disorder], but this isn't a diagnosis—let's talk about ways to seek clarity"), always urging professional help like helplines (e.g., beyondblue: 1300 22 4636).
    - Suggest practical steps gently, varying by context; highlight ICD-11 ideas as exploratory and non-diagnostic.
    - IMPORTANT: Do not mention or reference any 'Doc', 'Guideline', 'Section', scores, sentiments, sources, or internal labels. Integrate as innate knowledge. Use simple words, short sentences, and everyday language for easy reading.
    - Responses MUST be 30-100 words and complete (end with a full sentence); do not exceed 100 words to avoid cut-offs. Experiment with different styles, vocabulary, and phrasings for natural, varied flow. Start responses with diverse phrases like "That sounds tough...", "I'm here for you...", "Let's explore this together...", "I hear the pain in your words...", or "It's okay to feel this way..." to avoid repetition.
    - If the user's message appears to be a third-person description (e.g., about a patient or case), reinterpret it as a first-person personal story from the user themselves. Respond in a natural, empathetic dialogue style, as if they're sharing their own experiences directly (e.g., use "you" to address them, avoid referencing third parties unless specified).
    - Build on conversation history for personalization, referencing past details subtly to show continuity. Do not repeat phrases, suggestions, or structures from previous responses in the history—be creative and introduce new elements each time.
    - To increase variety: If the user repeats a message or similar sentiment multiple times, acknowledge the pattern empathetically (e.g., 'I notice this feeling keeps coming up—let's try a fresh way to approach it') and pivot to new elements like a specific question, analogy, unique coping tip, or helpline suggestion. Randomly incorporate structural variations: e.g., start with a question in one response, use bullet points for tips in another, share a brief relatable analogy in the next. Reference history more actively by summarizing or building on prior mentions (e.g., 'Last time you mentioned sadness; is it tied to that?').
    - For repetitive queries, escalate gently to immediate resources if appropriate (e.g., suggest a helpline number early).
    - Strictly avoid reusing any phrases or suggestions from conversation history—always innovate (e.g., if baths were suggested before, try nature walks instead).
    """


def trim_to_complete_sentence(text, max_words=100):
    """Trim text to max_words while ensuring it ends with a complete sentence."""
    words = text.split()
    if len(words) <= max_words:
        return text.strip()

    trimmed_text = ' '.join(words[:max_words])

    last_period = max(trimmed_text.rfind('.'), trimmed_text.rfind('.\n'), trimmed_text.rfind('. '))
    last_exclaim = max(trimmed_text.rfind('!'), trimmed_text.rfind('!\n'), trimmed_text.rfind('! '))
    last_question = max(trimmed_text.rfind('?'), trimmed_text.rfind('?\n'), trimmed_text.rfind('? '))
    last_sentence_end = max(last_period, last_exclaim, last_question)

    if last_sentence_end > 0:
        return trimmed_text[:last_sentence_end + 1].strip()

    # Fallback: end at a comma if possible
    last_comma = trimmed_text.rfind(',')
    if last_comma > len(trimmed_text) * 0.5:
        return trimmed_text[:last_comma + 1].strip()

    return trimmed_text.strip()


def update_repetition_count(query, last_query, repetition_count):
    """0 if this query differs from the last one, else the running repeat count + 1."""
    return (repetition_count + 1) if query == last_query else 0


def build_repetition_instruction(repetition_count):
    """Extra system-prompt instruction injected when the user repeats themselves.
    `repetition_count > 1` means at least 2 repeats (count starts at 0)."""
    if repetition_count <= 1:
        return ""
    instruction = (
        f"The user has repeated the same message {repetition_count + 1} times. "
        "Acknowledge this empathetically, avoid repeating past phrases, and pivot to a fresh "
        "approach like a new analogy, specific helpline, or unique question. Escalate to "
        "professional resources if repetition continues."
    )
    if repetition_count >= 2:  # 3+ total messages
        instruction += " Prioritize a helpline suggestion and a unique, creative analogy not used before."
    return instruction


def estimate_tokens(text: str) -> int:
    return len(text.split()) * 1.3 + 100  # Rough estimate + buffer


def retrieve_icd_matches(rewritten_query, k=4, score_threshold=0.7):
    """Search the ICD-11 index and format any high-confidence matches as a
    prompt section. Returns '' if nothing clears the threshold."""
    query_embedding = embedder.encode(rewritten_query, convert_to_tensor=True, device=device, normalize_embeddings=True, show_progress_bar=False)
    query_embedding_np = query_embedding.cpu().numpy().reshape(1, -1)
    distances, indices = icd_index.search(query_embedding_np, k=k)

    icd_infos = []
    for score, idx in zip(distances[0], indices[0]):
        if idx == -1 or score <= score_threshold:
            continue
        row = icd_df.iloc[idx]
        icd_infos.append(f"""
            Disorder Name: {row['fully_specified_name']}
            Disorder Code: {row['code']}
            Disorder symptoms: {row['description']}
            """)

    if not icd_infos:
        return ""
    return (
        f"\n\nPossible matching disorders from ICD-11:\n{''.join(icd_infos)}\n"
        "Remember, this is not a diagnosis; suggest professional help and integrate empathetically as non-diagnostic possibilities."
    )


def log_response_variety(previous_responses, response, enabled=False):
    """Optional diagnostic: logs TF-IDF cosine similarity between the last
    two assistant responses, to flag when responses start repeating. Off
    by default -- mirrors the `enable_variety_logging` flag from the
    original CLI/GUI/eval code, which was likewise always off."""
    if not enabled:
        return
    previous_responses.append(response)
    if len(previous_responses) < 2:
        return
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(previous_responses[-2:])
    similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    logger.info(f"Variety analysis: Cosine similarity between last two responses: {similarity:.2f} (lower is more varied)")
    if similarity > 0.7:
        logger.warning("High similarity detected; consider prompt tweaks for more variety.")


def generate_chatbot_reply(query, history, method='hyde', repetition_count=0,
                            max_context_tokens=8192, max_tokens=300, max_trim_words=130):
    """Core reply-generation logic shared by the CLI runner, GUI, and both
    evaluation harnesses below. Returns (response, retrieved_docs).

    `history` is prior turns BEFORE this query; this function does not
    mutate it or append to it -- the caller appends the new turn afterward.
    """
    rewritten_query = rewrite_query_with_history(query, history)
    icd_prompt_section = retrieve_icd_matches(rewritten_query)

    retrieved_docs = full_rag_workflow(rewritten_query, method=method)
    # No labels in the joined context, to avoid the model leaking "Doc"/"Section" mentions
    doc_context = "\n\n---\n\n".join(doc['content'] for doc in retrieved_docs)

    history_summary = (
        "Conversation history summary: " + " ".join(f"{msg['role']}: {msg['content'][:50]}..." for msg in history[-4:])
        if history else "No prior history."
    )

    repetition_instruction = build_repetition_instruction(repetition_count)

    user_prompt = f"""
        {history_summary}
        User's message: {query}
        Additional relevant guidelines (use these to inform suggestions if they fit the query):
        {doc_context}
        Respond empathetically, drawing from guidelines for specific ideas. If the query involves loss or grief, suggest personalized memorials or support resources. Vary your language from previous responses. Build on history for deeper personalization.
        {repetition_instruction}
        """ + icd_prompt_section

    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": user_prompt}]

    while sum(estimate_tokens(msg['content']) for msg in messages) > max_context_tokens:
        if len(history) <= 2:
            logger.warning("Context exceeds limit; proceeding.")
            break
        history = history[2:]  # Remove oldest pair
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": user_prompt}]
        logger.info(f"Truncated history to {len(history)} entries.")

    response = generate_with_llm(messages, max_tokens=max_tokens)
    response = trim_to_complete_sentence(response, max_words=max_trim_words)

    return response, retrieved_docs


## 11. CLI chatbot

Now a thin loop around `generate_chatbot_reply` -- all the prompt-building/retrieval logic lives in one place above.

In [ ]:
def run_chatbot(method: str = 'hyde', max_context_tokens: int = 8192):
    history = []  # List of dicts: [{'role': 'user', 'content': ...}, {'role': 'assistant', 'content': ...}]
    repetition_count = 0
    last_query = None
    previous_responses = []  # for optional variety logging
    enable_variety_logging = False

    print("Welcome to the Mental Health Chatbot. Type 'quit' to exit.")

    while True:
        query = input("You: ").strip()
        if query.lower() in ['quit', 'exit']:
            print("Chatbot: Goodbye! Take care.")
            break
        if not query:
            print("Chatbot: Please enter a message.")
            continue

        repetition_count = update_repetition_count(query, last_query, repetition_count)
        last_query = query

        try:
            response, _ = generate_chatbot_reply(
                query, history, method=method,
                repetition_count=repetition_count, max_context_tokens=max_context_tokens,
            )
            print("Chatbot:", response)

            history.append({"role": "user", "content": query})
            history.append({"role": "assistant", "content": response})

            log_response_variety(previous_responses, response, enabled=enable_variety_logging)
        except Exception as e:
            logger.error(f"Error in chatbot loop: {e}")
            print("Chatbot: Sorry, something went wrong. Please try again.")


run_chatbot(method='hyde')  # Change method as needed: 'original', 'multi', 'hyde'


## 12. GUI chatbot

Same consolidation -- the GUI class now delegates to `generate_chatbot_reply` instead of duplicating the whole prompt-building block inline.

In [ ]:
import tkinter as tk
from tkinter import Text, Scrollbar, Entry, Button, END, NORMAL, DISABLED


class MentalHealthChatbotGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Mental Health Chatbot")
        self.root.geometry("600x500")
        self.root.configure(bg="#17202A")

        self.history = []  # List of dicts: [{'role': 'user', 'content': ...}, {'role': 'assistant', 'content': ...}]
        self.max_context_tokens = 8192
        self.method = 'hyde'
        self.repetition_count = 0
        self.last_query = None
        self.previous_responses = []
        self.enable_variety_logging = False

        # Scrollbar for chat
        self.scrollbar = Scrollbar(self.root)
        self.scrollbar.place(relheight=0.85, relwidth=0.03, relx=0.0, rely=0.0)
        # Chat display area
        self.text_cons = Text(self.root, bg="#17202A", fg="#EAECEE", font="Helvetica 14", padx=5, pady=5, wrap="word", yscrollcommand=self.scrollbar.set)
        self.text_cons.place(relheight=0.85, relwidth=0.97, relx=0.03, rely=0.0)
        self.text_cons.config(state=DISABLED)
        self.scrollbar.config(command=self.text_cons.yview)
        self.text_cons.tag_config('user', justify='right', foreground="#AED6F1")  # Light blue for user
        self.text_cons.tag_config('bot', justify='left', foreground="#ABEBC6")  # Light green for bot
        # Entry for user message
        self.entry_msg = Entry(self.root, bg="#2C3E50", fg="#EAECEE", font="Helvetica 13")
        self.entry_msg.place(relwidth=0.74, relheight=0.06, rely=0.92, relx=0.011)
        self.entry_msg.focus()
        self.entry_msg.bind("<Return>", self.send_message)
        self.button_msg = Button(self.root, text="Send", font="Helvetica 10 bold", width=20, bg="#ABB2B9", command=self.send_message)
        self.button_msg.place(relx=0.77, rely=0.92, relheight=0.06, relwidth=0.22)

        self.append_message("Chatbot: Welcome to the Mental Health Chatbot. How can I help you today?\n")

    def append_message(self, message):
        self.text_cons.config(state=NORMAL)
        tag = 'user' if message.startswith("You:") else 'bot'
        self.text_cons.insert(END, message + "\n\n", tag)
        self.text_cons.config(state=DISABLED)
        self.text_cons.see(END)

    def send_message(self, event=None):
        query = self.entry_msg.get().strip()
        if not query:
            return
        if query.lower() in ['quit', 'exit']:
            self.append_message("Chatbot: Goodbye! Take care.")
            self.root.quit()
            return

        self.append_message(f"You: {query}")
        self.entry_msg.delete(0, END)

        self.repetition_count = update_repetition_count(query, self.last_query, self.repetition_count)
        self.last_query = query

        try:
            response, _ = generate_chatbot_reply(
                query, self.history, method=self.method,
                repetition_count=self.repetition_count,
                max_context_tokens=self.max_context_tokens,
                max_trim_words=150,
            )
            self.append_message(f"Chatbot: {response}")

            self.history.append({"role": "user", "content": query})
            self.history.append({"role": "assistant", "content": response})

            log_response_variety(self.previous_responses, response, enabled=self.enable_variety_logging)
        except Exception as e:
            logger.error(f"Error in chatbot: {e}")
            self.append_message("Chatbot: Sorry, something went wrong. Please try again.")


if __name__ == "__main__":
    root = tk.Tk()
    app = MentalHealthChatbotGUI(root)
    root.mainloop()


## 13. Evaluation: prep DSM-5 test set

Uses the anonymized DSM-5-TR clinical cases as a test set -- case *description* as the query, case *diagnosis* as the reference/ground truth.

In [ ]:
import pandas as pd
import numpy as np
import re
from bert_score import score as bert_score
from nltk.util import ngrams

test_df = pd.read_csv('./rag_system/dsm5/dsm5_testing_anonymized.csv')  # Columns: case, description, discussion, diagnosis

all_test_queries = test_df['description'].tolist()
all_ground_truth_diagnoses = test_df['diagnosis'].tolist()

k = 10
test_queries = all_test_queries[:k]
ground_truth_diagnoses = all_ground_truth_diagnoses[:k]


## 14. Evaluation: generate responses for the test set

Single-turn simulation (no persisted history across test queries -- each one is independent), built on the same shared `generate_chatbot_reply`.

In [ ]:
def simulate_chatbot_single(query, method='hyde'):
    """Single-turn simulation for the RAG retrieval evaluation below --
    each test query is independent, so history is always empty."""
    try:
        response, retrieved_docs = generate_chatbot_reply(query, history=[], method=method)
    except Exception as e:
        logger.error(f"Error in chatbot loop: {e}")
        response, retrieved_docs = "Sorry, something went wrong. Please try again.", []
    return response, retrieved_docs


retrieved_contents = []
generated_responses = []
for query in test_queries:
    response, retrieved_docs = simulate_chatbot_single(query, method="hyde")  # Change method as needed: 'original', 'multi', 'hyde'
    retrieved_contents.append([doc['content'] for doc in retrieved_docs])
    generated_responses.append(response)

import pickle
with open('./rag_system/testing_reference_data/test_retrieved_contents_discription.pkl', 'wb') as f:
    pickle.dump(retrieved_contents, f)
with open('./rag_system/testing_reference_data/test_generated_responses_discription.pkl', 'wb') as f:
    pickle.dump(generated_responses, f)


## 15. Evaluation: RAG retrieval quality

nDCG@5 (using cosine similarity as the relevance proxy) plus semantic relevance of retrieved content and generated responses against the query.

In [ ]:
import pickle
with open('./rag_system/testing_reference_data/test_retrieved_contents_discription.pkl', 'rb') as f:
    retrieved_contents = pickle.load(f)
with open('./rag_system/testing_reference_data/test_generated_responses_discription.pkl', 'rb') as f:
    generated_responses = pickle.load(f)

with open('./rag_system/testing_reference_data/test_retrieved_contents_discription.txt', 'w', encoding='utf-8') as f:
    for i, contents in enumerate(retrieved_contents):
        f.write(f"Query {i+1}:\n")
        for j, content in enumerate(contents):
            f.write(f"Retrieved {j+1}:\n{content}\n\n")
        f.write("\n" + "=" * 50 + "\n\n")

with open('./rag_system/testing_reference_data/test_generated_responses_discription.txt', 'w', encoding='utf-8') as f:
    for i, response in enumerate(generated_responses):
        f.write(f"Query {i+1}:\n")
        f.write(f"Generated Response:\n{response}\n\n")
        f.write("\n" + "=" * 50 + "\n\n")


def compute_ndcg(relevances, k=5):
    """nDCG@K given a list of relevance scores."""
    if not relevances:
        return 0
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances[:k]))
    ideal_relevances = sorted(relevances, reverse=True)[:k]
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevances))
    return dcg / idcg if idcg > 0 else 0


def cosine_sim(a, b):
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0
    return np.dot(a.flatten(), b.flatten()) / (norm_a * norm_b)


rag_ndcgs = []
retrieved_semantic_scores = []
for i, query in enumerate(test_queries):
    retrieved = retrieved_contents[i]
    if not retrieved:
        rag_ndcgs.append(0)
        retrieved_semantic_scores.append(0)
        continue

    query_emb = embedder.encode(query, convert_to_tensor=True, device=device, show_progress_bar=False).cpu().numpy()
    retrieved_embs = embedder.encode(retrieved, convert_to_tensor=True, device=device, show_progress_bar=False).cpu().numpy()

    similarities = [cosine_sim(query_emb, emb) for emb in retrieved_embs]
    rag_ndcgs.append(compute_ndcg(similarities, k=5))
    retrieved_semantic_scores.append(np.mean(similarities))

avg_ndcg = np.mean(rag_ndcgs)
avg_retrieved_sem = np.mean(retrieved_semantic_scores)

generated_semantic_scores = []
for i, query in enumerate(test_queries):
    query_emb = embedder.encode(query, convert_to_tensor=True, device=device, show_progress_bar=False).cpu().numpy()
    gen_emb = embedder.encode(generated_responses[i], convert_to_tensor=True, device=device, show_progress_bar=False).cpu().numpy()
    generated_semantic_scores.append(cosine_sim(query_emb, gen_emb))

avg_generated_sem = np.mean(generated_semantic_scores)

print("RAG evaluation results:")
print(f"Average nDCG@5 for RAG: {avg_ndcg}")
print(f"Average semantic relevance (cosine) for retrieved vs. query: {avg_retrieved_sem}")
print(f"Average semantic relevance (cosine) for generated vs. query: {avg_generated_sem}")


## 16. Evaluation: chatbot response quality

Empathy proxy (VADER sentiment), readability (Flesch), perplexity (GPT-2, as a coherence proxy), toxicity (Detoxify), and LLM-as-judge helpfulness.

In [ ]:
import textstat
from textblob import TextBlob
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
from detoxify import Detoxify
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import numpy as np
from nltk.util import ngrams

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')
detox_model = Detoxify('original')
sia = SentimentIntensityAnalyzer()


def compute_perplexity(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = gpt2_model(**inputs, labels=inputs["input_ids"])
    return torch.exp(outputs.loss).item()


empathy_scores = []  # VADER compound as proxy (or LLM-as-judge)
helpfulness_scores = []
readability_scores = []
perplexity_scores = []
toxicity_scores = []

for i, response in enumerate(generated_responses):
    empathy_scores.append(sia.polarity_scores(response)['compound'])
    readability_scores.append(textstat.flesch_reading_ease(response))
    perplexity_scores.append(compute_perplexity(response))
    toxicity_scores.append(detox_model.predict(response)['toxicity'])

    judge_prompt = f"""On a scale from 1 to 10, rate how helpful and empathetic this response is for the user's query. Output only the number, nothing else.
        Query: {test_queries[i]}
        Response: {response}"""
    judge_response = generate_with_llm([{"role": "user", "content": judge_prompt}], max_tokens=10)
    try:
        helpfulness = float(re.search(r'\d+(\.\d+)?', judge_response).group())
    except Exception:
        helpfulness = 5.0  # Default if parsing fails
    helpfulness_scores.append(helpfulness)

print(f"Average Empathy/Sentiment (VADER): {np.mean(empathy_scores)}")
print(f"Average Readability (Flesch): {np.mean(readability_scores)}")
print(f"Average Perplexity (lower better): {np.mean(perplexity_scores)}")
print(f"Average Toxicity (lower better): {np.mean(toxicity_scores)}")
print(f"Average Helpfulness (LLM Judge): {np.mean(helpfulness_scores)}")


## 17. Evaluation: response diversity under repetition

Simulates a user sending the **same** query several times in a row (as a distressed/repetitive user might), and checks whether responses stay varied (Distinct-2) instead of repeating themselves.

Renamed from the original `simulate_chatbot(test_queries, ...)` to `simulate_repeated_query(query, ...)` -- the old name took a *single* query string despite the plural parameter name (it did `test_queries = [test_queries] * iterations` internally), which was confusing to read. Behavior is unchanged, just clearer naming, and it now calls the shared `generate_chatbot_reply` instead of its own copy of the prompt-building logic.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os
import json


def load_test_queries(file_path='./rag_system/testing_reference_data/repeated_test_queries.json'):
    """Flat list of query strings (JSON list)."""
    with open(file_path, 'r') as f:
        return json.load(f)


def simulate_repeated_query(query, method: str = 'hyde', iterations: int = 3, max_context_tokens: int = 8192):
    """Ask the SAME query `iterations` times in a row, carrying history
    forward between iterations, and collect the generated responses."""
    history = []
    generated_responses = []

    for i in range(iterations):
        repetition_count = i  # every iteration is a literal repeat of the same query
        try:
            response, _ = generate_chatbot_reply(
                query, history, method=method,
                repetition_count=repetition_count, max_context_tokens=max_context_tokens,
            )
            print(f"You: {query}")
            print(f"Chatbot: {response}")

            history.append({"role": "user", "content": query})
            history.append({"role": "assistant", "content": response})
            generated_responses.append(response)
        except Exception as e:
            logger.error(f"Error in chatbot loop: {e}")

    return generated_responses


def distinct_n(texts, n=2):
    """Distinct-n diversity metric for a list of texts."""
    all_ngrams = []
    for text in texts:
        tokens = text.split()
        all_ngrams.extend(list(ngrams(tokens, n)))
    if not all_ngrams:
        return 0
    return len(set(all_ngrams)) / len(all_ngrams)


test_queries = load_test_queries()

whole_generated_response = {}
whole_distinct_n = {}
for query in test_queries:
    responses = simulate_repeated_query(query, method='hyde', iterations=3)
    whole_generated_response[query] = responses
    whole_distinct_n[query] = distinct_n(responses, n=2)

print("Diversity (Distinct-2) results:")
general_diversity = 0
for query, diversity in whole_distinct_n.items():
    print(f"Query: {query}\nDistinct-2: {diversity}\n")
    general_diversity += diversity
general_diversity /= len(whole_distinct_n)
print(f"Average Distinct-2 across all queries: {general_diversity}")

output_dir = "./rag_system/testing_reference_data"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "repetition_test.txt")

with open(output_file, "w") as f:
    for query, responses in whole_generated_response.items():
        f.write(f"Query: {query}\n")
        for i, response in enumerate(responses):
            f.write(f"Response {i+1}: {response}\n")
        f.write("\n" + "=" * 50 + "\n\n")
